In [ ]:
# %load_ext autoreload
# %autoreload 2

In [2]:
# fetch who pdfs push embeddings to db (~90 min depends on host)
# re-try if Totally fetched/scraped 0 .pdf links!, then check db by adminer
!python assistant/pipeline.py

2025-09-09 21:49:35,193 [INFO] db - text_chunks Table created if not exist!
2025-09-09 21:49:44,824 [INFO] scraper - Loaded page: 1 found: 32 .pdf in set.
2025-09-09 21:49:55,875 [INFO] scraper - Totally fetched/scraped 32 .pdf links!
2025-09-09 21:53:44,427 [INFO] db - (Group 1) In table: text_chunks All Embedding/Vector data added!
2025-09-09 21:53:44,429 [INFO] __main__ - Ingested https://iris.who.int/bitstream/handle/10665/375060/WHO-EURO-2025-8926-48698-76066-eng.pdf?sequence=2 (Group 1)
2025-09-09 21:57:22,204 [INFO] db - (Group 2) In table: text_chunks All Embedding/Vector data added!
2025-09-09 21:57:22,213 [INFO] __main__ - Ingested https://iris.who.int/bitstream/handle/10665/381726/WHO-EURO-2025-12421-52195-80181-eng.pdf?sequence=1 (Group 2)
2025-09-09 22:01:31,230 [INFO] db - (Group 3) In table: text_chunks All Embedding/Vector data added!
2025-09-09 22:01:31,236 [INFO] __main__ - Ingested https://iris.who.int/bitstream/handle/10665/381950/9789289062329-eng.pdf?sequence=2 (G

In [3]:
!python assistant/cli.py --help

Usage: cli.py [OPTIONS] COMMAND [ARGS]...

  Assistant CLI

Options:
  --help  Show this message and exit.

Commands:
  ask  Query the assistant with Retrieval-Augmented Generation (RAG) search.


2025-09-09 23:20:16,198 [INFO] faiss.loader - Loading faiss with AVX2 support.
2025-09-09 23:20:16,241 [INFO] faiss.loader - Successfully loaded faiss with AVX2 support.


In [4]:
# Ask questions (ollama ~11 min)
!python assistant/cli.py ask -q "adult population in Ukraine"


[Best Match: ID 390 link https://iris.who.int/bitstream/handle/10665/382045/WHO-EURO-2025-6904-46670-80597-eng.pdf?sequence=1]

=== Assistant Answer ===

Based on the available data and considering demographic variations across different regions, approximately 30% of adult Ukrainians could have at least one family member who has tried to access general health services since February 24th, 2022. Specifically referring back to your context (as mentioned in paragraph B), it is understood that this percentage was estimated based on preliminary findings from the study team's work, which included various data sources concerning Ukraine's adult population distribution by age and gender as well as rural versus urban residency across macroregions.


2025-09-09 23:20:28,919 [INFO] faiss.loader - Loading faiss with AVX2 support.
2025-09-09 23:20:28,962 [INFO] faiss.loader - Successfully loaded faiss with AVX2 support.
2025-09-09 23:30:40,197 [INFO] openai._base_client - Retrying request to /chat/completions in 0.483035 seconds
2025-09-09 23:31:24,975 [INFO] httpx - HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"


In [6]:
# Ask questions
!streamlit run app_st.py

^C


In [ ]:
from assistant import config
config.SETTINGS.as_dict

In [ ]:
from assistant import pipeline

In [ ]:
# One time embeddings to add db
pipeline.ingest()

In [ ]:
from assistant import embeddings

query_vector = embeddings.generate_embedding("adult population in Ukraine")
len(query_vector), query_vector[:10]

In [ ]:
from assistant import search

pg = search.search_scipy_cosine(query_vector, top_n=5)
pg

In [ ]:
search.display_results(pg, "Scipy Cosine", score_label="similarity")

In [ ]:
import requests

payload = {
    "prompt": "hello",
    "model": config.SETTINGS.MODEL_EMBED,  # Ollama model
}
# POST request to Ollama embeddings endpoint
r = requests.post(
    url=f"{config.SETTINGS.BASE_URL.replace("v1", '')}api/embeddings",  # Ollama API endpoint
    json=payload,
    # timeout=60  # Optional: consider adding a timeout
)
r.raise_for_status()  # Raise exception if HTTP error
emb = r.json().get("embedding", [])
emb

In [ ]:
import requests

r = requests.post(
    url=f"{config.SETTINGS.BASE_URL}/chat/completions",
    headers={"Authorization": f"Bearer {config.SETTINGS.API_KEY}"},
    json={
        "model": config.SETTINGS.MODEL_CHAT,
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant. Use the provided context when answering. "
                    "For that you get additional information from a database. It's always a piece of text. "
                    "Please consider this text in your answer. Give a detailed answer."
                ),
            },
            {
                "role": "assistant",
                "content": "Context (from database): with high prices being the main concern. Pharmacy closures and medicine shortages were more common in front- line areas. Awareness of the Affordable Medicines Programme (AMP) increased to 62%, though only 24% of those aware used it. Vaccination access remained stable, with low COVID- 19 vaccine dema...",
            },
            {"role": "user", "content": "adult population in Ukraine"},
        ],
        "temperature": 0.7,
    },
    # timeout=120  # seconds
)
r.json().get("choices", [{"message": {"content": ""}}])[0]["message"]["content"]  # parse the JSON body